<div style='width:100%; display:flex; justify-content:space-between; align-items:center; margin: 1em 0;'>
  <a href='15.%20error_handling_and_logging.ipynb' style='padding:6px 14px; background:#007bff; color:white; border-radius:4px; text-decoration:none; font-weight:bold;'>&#8592; Chapter 15: Error Handling</a>
  <a href='../TOC.md' style='padding:6px 14px; background:#6c757d; color:white; border-radius:4px; text-decoration:none; font-weight:bold;'>&#128218; Table of Contents</a>
  <a href='../7.%20deployment_and_production/17.%20caching_and_performance.ipynb' style='padding:6px 14px; background:#28a745; color:white; border-radius:4px; text-decoration:none; font-weight:bold;'>Chapter 17: Caching &#8594;</a>
</div>

# Chapter 16: Testing Flask Applications -- Confidence at Scale

> *"Imagine deploying a new feature and discovering you've broken login for 10,000 users. A test suite catches that in seconds. Tests are your safety net -- they let you change code confidently, knowing you'll be alerted the moment something breaks."*

## 🎯 The Big Picture

Software without tests is like a bridge without load testing -- it might be fine, or it might collapse under pressure. Tests let you:

- **Refactor fearlessly** -- change internals knowing the behaviour is preserved
- **Catch regressions** -- know immediately when a new feature breaks an old one
- **Document behaviour** -- tests describe what your code is supposed to do
- **Deploy with confidence** -- CI/CD runs tests before every deploy

Flask provides a **test client** that acts as a fake browser: it makes HTTP requests directly to your app in memory -- no network, no browser, just Python.

```text
Test code
   |
   v
app.test_client()  -->  Flask routes  -->  Response object
                                               |
                               assert response.status_code == 200
                               assert b"Welcome" in response.data
```

> ❓ **Why use a fake client instead of a real browser?** The test client sends requests directly to your app in memory — no network, no server startup. Tests run in milliseconds and work in CI without any extra infrastructure.

## 🧠 Core Concepts: The "Why"

### The Testing Pyramid

Treat this section as a feedback loop: define an expectation, exercise the behavior, compare the result, and use the difference to learn. That framing keeps testing ideas concrete instead of procedural.

```text
        /\
       /E2E\       End-to-End (few, slow, test whole system)
      /------\
     / Integr. \    Integration (some, medium, test multiple units)
    /------------\
   /   Unit Tests  \  Unit (many, fast, test single functions)
  /________________\
```

| Layer | What it tests | Speed | Count |
|-------|---------------|-------|-------|
| **Unit** | Single function, no side effects | Very fast | Many |
| **Integration** | Multiple components together | Medium | Some |
| **End-to-End** | Entire user flow in real browser | Slow | Few |

For Flask apps, integration tests via the test client give the best return on investment.
### The Flask Test Client

The test client is a fake browser that talks to your Flask app without a network:
- No HTTP round-trip -- calls routes directly in memory
- Can send GET, POST, PUT, DELETE requests
- Receives Response objects with status_code, data, headers
- Can manage cookies/sessions (for login state)

## ⚡ Syntax & First Usage

### Your First Flask Test

Testing concepts stick better when you remember the purpose behind each step: create evidence that the behavior you care about still works, and learn quickly when it stops working.

In [ ]:

# Simplest possible Flask test -- no pytest needed to understand the concept

from flask import Flask

app = Flask(__name__)

@app.route("/")
def index():
    return "<h1>Welcome to My App</h1>"

@app.route("/about")
def about():
    return "<h1>About Page</h1>"

# Create a test client
client = app.test_client()

# Make requests just like a browser would
response = client.get("/")
print(f"Status: {response.status_code}")      # 200
print(f"Data type: {type(response.data)}")    # bytes
print(f"Content: {response.data}")            # b"<h1>Welcome..."

# Assertions
assert response.status_code == 200
assert b"Welcome" in response.data
print()

response_about = client.get("/about")
assert response_about.status_code == 200
assert b"About" in response_about.data

response_missing = client.get("/nonexistent-page")
assert response_missing.status_code == 404

print("All assertions passed! Basic test client works.")
print()
print("Key: response.data is BYTES, so compare with b'' strings, not regular strings")


### pytest Setup for Flask -- the `conftest.py` File

In [ ]:

# conftest.py -- pytest reads this automatically, no imports needed
# Place at the root of your tests/ directory

conftest_code = """
import pytest
from app import create_app, db

@pytest.fixture
def app():
    # Create app with test configuration
    app = create_app({
        "TESTING": True,
        "SQLALCHEMY_DATABASE_URI": "sqlite:///:memory:",  # in-memory, no files
        "WTF_CSRF_ENABLED": False,                        # disable CSRF for tests
        "SECRET_KEY": "test-secret-key"
    })

    with app.app_context():
        db.create_all()           # create all tables fresh
        yield app                 # run the test
        db.drop_all()             # clean up after test

@pytest.fixture
def client(app):
    return app.test_client()

@pytest.fixture
def auth_client(client):
    # Log in a test user before the test
    client.post("/auth/register", data={
        "username": "testuser",
        "email": "test@example.com",
        "password": "testpass123"
    })
    client.post("/auth/login", data={
        "email": "test@example.com",
        "password": "testpass123"
    })
    return client
"""

print("conftest.py structure:")
print(conftest_code)
print()
print("pytest discovers fixtures automatically from conftest.py")
print("Any test that needs 'client' gets a fresh test client")
print("Any test that needs 'auth_client' gets a logged-in client")
print("The 'app' fixture creates a fresh database for EACH test")


## 🔬 Deep Dive

### Testing GET Routes

Testing concepts stick better when you remember the purpose behind each step: create evidence that the behavior you care about still works, and learn quickly when it stops working.

In [ ]:

# Testing GET routes: status codes, content, template context

get_tests = """
import pytest
from app import create_app, db

# Tests for the homepage
def test_homepage_loads(client):
    response = client.get("/")
    assert response.status_code == 200

def test_homepage_contains_expected_content(client):
    response = client.get("/")
    assert b"Flask Blog" in response.data        # site name
    assert b"Latest Posts" in response.data      # section heading

def test_homepage_returns_html(client):
    response = client.get("/")
    assert response.content_type == "text/html; charset=utf-8"

def test_nonexistent_route_returns_404(client):
    response = client.get("/this-does-not-exist")
    assert response.status_code == 404

def test_about_page(client):
    response = client.get("/about")
    assert response.status_code == 200
    assert b"About" in response.data
"""

print("GET route tests:")
print(get_tests)
print()
print("Run with: pytest tests/ -v")
print()
print("Useful response attributes:")
attrs = [
    ("response.status_code", "int, e.g. 200"),
    ("response.data",         "bytes -- the response body"),
    ("response.json",         "dict -- if response is JSON"),
    ("response.headers",      "dict-like -- response headers"),
    ("response.content_type", "str -- e.g. 'text/html'"),
    ("response.location",     "str -- redirect URL (if 302)"),
]
for attr, desc in attrs:
    print(f"  {attr:<30} {desc}")


### Testing POST Routes: Form Submissions

In [ ]:

post_tests = """
def test_create_post(auth_client, app):
    # Make a POST request with form data
    response = auth_client.post("/posts/create", data={
        "title": "My Test Post",
        "body":  "This is the body of the test post.",
        "submit": True
    }, follow_redirects=True)   # follow the redirect after successful create

    assert response.status_code == 200
    assert b"My Test Post" in response.data   # should appear on the page

    # Verify the post was saved to the database
    with app.app_context():
        from app.models import Post
        post = Post.query.filter_by(title="My Test Post").first()
        assert post is not None
        assert post.body == "This is the body of the test post."


def test_create_post_requires_login(client):
    # Without login, should redirect to login page
    response = client.post("/posts/create", data={
        "title": "Unauthorized Post",
        "body": "Should not be saved"
    })
    assert response.status_code == 302        # redirect
    assert "/login" in response.location      # to login page


def test_create_post_validates_required_fields(auth_client):
    # Empty title should fail validation
    response = auth_client.post("/posts/create", data={
        "title": "",
        "body": "Body without title"
    }, follow_redirects=True)
    assert b"This field is required" in response.data
"""

print("POST route tests:")
print(post_tests)
print()
print("Key patterns:")
print("  follow_redirects=True -- follow 302 redirects automatically")
print("  data={} -- simulates form submission")
print("  Check DB state to verify persistence (not just the response)")


### Testing Authentication

In [ ]:

auth_tests = """
def test_login_with_valid_credentials(client, app):
    # First create a user
    with app.app_context():
        user = User(username="alice", email="alice@example.com")
        user.set_password("correctpassword")
        db.session.add(user)
        db.session.commit()

    # Try logging in
    response = client.post("/auth/login", data={
        "email": "alice@example.com",
        "password": "correctpassword"
    }, follow_redirects=True)

    assert response.status_code == 200
    assert b"Welcome back" in response.data    # success flash message


def test_login_with_wrong_password(client, app):
    with app.app_context():
        user = User(username="bob", email="bob@example.com")
        user.set_password("correctpassword")
        db.session.add(user)
        db.session.commit()

    response = client.post("/auth/login", data={
        "email": "bob@example.com",
        "password": "wrongpassword"
    }, follow_redirects=True)

    assert b"Invalid credentials" in response.data


def test_protected_route_redirects_when_not_logged_in(client):
    response = client.get("/posts/create")
    assert response.status_code == 302
    assert "login" in response.location


def test_logout(auth_client):
    # auth_client is already logged in (from conftest fixture)
    response = auth_client.get("/auth/logout", follow_redirects=True)
    assert response.status_code == 200
    assert b"Logged out" in response.data
"""

print("Authentication tests:")
print(auth_tests)


### Testing the REST API (JSON endpoints)

Treat this section as a feedback loop: define an expectation, exercise the behavior, compare the result, and use the difference to learn. That framing keeps testing ideas concrete instead of procedural.

In [ ]:

api_tests = """
import json

def test_api_get_posts(client):
    response = client.get("/api/posts",
        headers={"Accept": "application/json"}
    )
    assert response.status_code == 200
    assert response.content_type == "application/json"
    data = response.json
    assert "posts" in data
    assert isinstance(data["posts"], list)


def test_api_create_post_requires_auth(client):
    response = client.post("/api/posts",
        json={"title": "New Post", "body": "Content"},  # json= sends JSON
        headers={"Accept": "application/json"}
    )
    assert response.status_code == 401     # Unauthorized


def test_api_create_post_authenticated(client, app):
    # Get a valid JWT token first
    with app.app_context():
        user = User(username="api_user", email="api@ex.com")
        user.set_password("pass")
        db.session.add(user)
        db.session.commit()
        token = user.generate_auth_token()

    response = client.post("/api/posts",
        json={"title": "API Post", "body": "Via API"},
        headers={
            "Authorization": f"Bearer {token}",
            "Content-Type": "application/json"
        }
    )
    assert response.status_code == 201    # Created
    assert response.json["title"] == "API Post"
"""

print("API tests:")
print(api_tests)
print()
print("client.post(json={...}) automatically:")
print("  - Serializes dict to JSON string")
print("  - Sets Content-Type: application/json")
print("  - No need for json.dumps() or explicit headers")


### Testing Database Operations

In [ ]:

db_tests = """
def test_post_is_saved_to_database(auth_client, app):
    auth_client.post("/posts/create", data={
        "title": "DB Test Post",
        "body": "Testing database persistence"
    }, follow_redirects=True)

    with app.app_context():
        from app.models import Post
        post = Post.query.filter_by(title="DB Test Post").first()
        assert post is not None
        assert post.body == "Testing database persistence"


def test_post_delete_removes_from_db(auth_client, app):
    # Create a post first
    with app.app_context():
        from app.models import Post
        post = Post(title="To Delete", body="Will be deleted", author_id=1)
        db.session.add(post)
        db.session.commit()
        post_id = post.id

    # Delete via API
    auth_client.delete(f"/api/posts/{post_id}")

    # Verify it's gone
    with app.app_context():
        from app.models import Post
        deleted = Post.query.get(post_id)
        assert deleted is None


def test_user_model_password_hashing(app):
    with app.app_context():
        user = User(username="hash_test")
        user.set_password("mypassword")
        db.session.add(user)
        db.session.commit()

        assert user.password_hash != "mypassword"       # never stored plaintext
        assert user.check_password("mypassword") is True
        assert user.check_password("wrongpassword") is False
"""

print("Database tests:")
print(db_tests)
print()
print("Key pattern: always use 'with app.app_context():' when accessing DB")
print("The in-memory SQLite DB is created fresh for each test (via conftest fixture)")


### Test Isolation: Why Each Test Must Be Independent

Testing concepts stick better when you remember the purpose behind each step: create evidence that the behavior you care about still works, and learn quickly when it stops working.

In [ ]:

# The danger of shared state between tests

bad_example = """
# BROKEN -- tests depend on each other (shared state)
posts = []                # module-level shared state!

def test_create_post():
    posts.append({"title": "Post 1"})
    assert len(posts) == 1   # passes when run alone

def test_count_posts():
    assert len(posts) == 0   # FAILS when run after test_create_post!
    # But passes when run alone. "Flaky test" -- passes sometimes, fails others.
"""

good_example = """
# CORRECT -- each test sets up its own state
def test_create_post(client, app):
    # Fresh DB from the 'app' fixture (creates + drops tables each test)
    response = client.post("/posts/create", data={...})
    with app.app_context():
        assert Post.query.count() == 1

def test_count_posts(client, app):
    # Also gets a fresh empty DB -- completely independent
    with app.app_context():
        assert Post.query.count() == 0  # Always passes
"""

flaky_signs = [
    "Test passes when run alone, fails in full suite",
    "Tests pass in one order, fail in another",
    "Tests fail only on CI, not locally",
    "One failing test causes multiple subsequent failures",
]

print("Bad (shared state):", bad_example)
print("Good (isolated):", good_example)
print()
print("Signs of non-isolated (flaky) tests:")
for sign in flaky_signs:
    print(f"  - {sign}")

print()
print("Fix: use fixtures that create and tear down state per-test.")
print("The 'app' fixture in conftest.py creates a fresh DB for every test.")


### Code Coverage

In [ ]:

coverage_info = """
Code coverage measures which lines of your code are executed during tests.

Installation:
    pip install pytest-cov

Basic usage:
    pytest --cov=app                        # print coverage to terminal
    pytest --cov=app --cov-report=html      # generate HTML report in htmlcov/
    pytest --cov=app --cov-report=term-missing  # show which lines are missed

Example output:
    Name                        Stmts   Miss  Cover
    -----------------------------------------------
    app/__init__.py                30      0   100%
    app/auth/routes.py             85     12    86%
    app/posts/routes.py            60      5    92%
    app/models.py                  45      0   100%
    -----------------------------------------------
    TOTAL                         220     17    92%

Coverage goal: 80%+ is a reasonable target for most projects.
100% coverage does NOT mean bug-free -- it means every line ran at least once.

What to do with uncovered lines:
    - Look at the missing lines (--cov-report=term-missing)
    - Write tests that exercise those paths
    - Mark intentionally untested code with # pragma: no cover
"""

print(coverage_info)

config_ini = """
# pytest.ini or setup.cfg -- configure coverage defaults
[pytest]
addopts = --cov=app --cov-report=term-missing
testpaths = tests

# .coveragerc
[report]
omit =
    */migrations/*
    */config.py
    */tests/*
"""
print("Configuration files:")
print(config_ini)


### ⚖️ Comparison: unittest vs pytest

In [ ]:

unittest_version = """
# unittest (standard library)
import unittest
from app import create_app, db

class TestPosts(unittest.TestCase):
    def setUp(self):
        self.app = create_app({"TESTING": True, "SQLALCHEMY_DATABASE_URI": "sqlite:///:memory:"})
        self.client = self.app.test_client()
        with self.app.app_context():
            db.create_all()

    def tearDown(self):
        with self.app.app_context():
            db.drop_all()

    def test_homepage(self):
        response = self.client.get("/")
        self.assertEqual(response.status_code, 200)
        self.assertIn(b"Welcome", response.data)

if __name__ == "__main__":
    unittest.main()
"""

pytest_version = """
# pytest (much cleaner with fixtures)
def test_homepage(client):         # 'client' comes from conftest.py
    response = client.get("/")
    assert response.status_code == 200
    assert b"Welcome" in response.data
"""

comparison = """
+------------------+-------------------------------------------+---------------------------+
| Feature          | unittest                                  | pytest                    |
+------------------+-------------------------------------------+---------------------------+
| Boilerplate      | Heavy (setUp, tearDown, self.assert*)     | Minimal (just functions)  |
| Fixtures         | setUp/tearDown per class                  | Reusable, composable      |
| Assertions       | self.assertEqual(a, b)                    | assert a == b             |
| Parametrize      | Manual test methods                       | @pytest.mark.parametrize  |
| Output           | . or F                                    | Colorful, verbose (-v)    |
| Plugins          | Limited                                   | Rich ecosystem            |
+------------------+-------------------------------------------+---------------------------+
"""

print("unittest version:")
print(unittest_version)
print("pytest version:")
print(pytest_version)
print("Comparison:")
print(comparison)
print("Verdict: pytest is the modern standard for Flask apps.")


### ⚖️ In-Memory SQLite vs Real Database for Tests

In [ ]:

comparison_code = """
+---------------------+--------------------------------+--------------------------------+
| Aspect              | In-memory SQLite               | Production DB (PostgreSQL)     |
+---------------------+--------------------------------+--------------------------------+
| Speed               | Very fast (no disk I/O)        | Slower                         |
| Isolation           | Perfect (fresh each test)      | Requires careful setup         |
| Similarity to prod  | Some differences in SQL        | Exactly like production        |
| CI/CD setup         | Zero setup needed              | Needs DB service configured    |
| Dialect differences | Missing some PostgreSQL features | None                         |
| Best for            | 90% of tests                   | DB-specific edge cases         |
+---------------------+--------------------------------+--------------------------------+
"""

recommendation = """
Strategy: use SQLite in-memory for 90%+ of tests (unit + most integration).
Run a separate CI job against real PostgreSQL for DB-specific tests.

Things that differ between SQLite and PostgreSQL:
  - Array columns (PostgreSQL has ARRAY type, SQLite does not)
  - Full-text search (different syntax)
  - JSONB operations (PostgreSQL-specific)
  - Strict mode for data types

For these cases, use pytest marks to separate them:

    @pytest.mark.postgres
    def test_full_text_search(client):
        ...

Then run: pytest -m "not postgres"   # fast tests only
          pytest -m postgres         # PostgreSQL-specific (in CI with real DB)
"""

print(comparison_code)
print(recommendation)


## 🧪 What If?

### What if tests share state and become flaky?

In [ ]:

# Demonstrating flaky test diagnosis and fix

flaky_scenario = """
The Bug:
--------
    # In conftest.py (WRONG -- same app instance for all tests)
    @pytest.fixture(scope="module")   # <-- "module" scope is the problem
    def app():
        app = create_app({"TESTING": True, "SQLALCHEMY_DATABASE_URI": "sqlite:///:memory:"})
        with app.app_context():
            db.create_all()
            yield app
            db.drop_all()

    # test_1.py
    def test_create_user(app, client):
        client.post("/auth/register", data={"username": "alice", ...})
        # Test passes. Alice is now in the shared DB.

    # test_2.py
    def test_register_unique_username(client):
        response = client.post("/auth/register", data={"username": "alice", ...})
        assert response.status_code == 400    # expects username taken error
        # BUT: if test_1 runs first, alice already exists -> passes
        # If test_2 runs first (e.g., pytest --randomly), alice doesn't exist -> FAILS

The Fix:
--------
    @pytest.fixture(scope="function")  # <-- default, creates fresh DB per test
    def app():
        app = create_app({"TESTING": True, "SQLALCHEMY_DATABASE_URI": "sqlite:///:memory:"})
        with app.app_context():
            db.create_all()
            yield app
            db.drop_all()
"""

print(flaky_scenario)
print("Tools to detect flaky tests:")
print("  pytest-randomly  -- randomize test order to expose ordering dependencies")
print("  pytest -p no:randomly  -- run in fixed order when debugging")
print("  pytest --lf  -- run only last-failed tests")


### What if DEBUG=True in tests? What if coverage is 0% for a critical module?

In [ ]:

print("=== DEBUG=True in tests ===")
debug_issue = """
When TESTING=True, Flask disables its error handling so exceptions propagate
to the test code (which is what you want -- see the actual error).

When DEBUG=True (even in tests):
  - Werkzeug's reloader may interfere with test collection
  - Some production error paths that are only triggered in non-debug mode won't be tested
  - May see different behaviour in tests than in production

Best practice in TestingConfig:
    DEBUG = False
    TESTING = True   # this is what disables error catching, not DEBUG
"""
print(debug_issue)

print()
print("=== Coverage is 0% for a critical module ===")
coverage_issue = """
If pytest --cov shows 0% for app/payments/routes.py, it means:
  - None of your tests hit those routes AT ALL
  - Any bug in that module will reach production undetected

Steps to fix:
1. Find the uncovered file:
       pytest --cov=app --cov-report=term-missing

2. Write tests for the missing module:
       # tests/test_payments.py
       def test_checkout_page_loads(auth_client):
           response = auth_client.get("/payments/checkout")
           assert response.status_code == 200

3. Protect with a coverage threshold:
       # pytest.ini
       [pytest]
       addopts = --cov=app --cov-fail-under=80

       Now pytest fails if coverage drops below 80%.
       Use this in CI to prevent coverage regression.
"""
print(coverage_issue)


## 🚀 Real-World Project Link

In our **Blog Application**, the test suite is organized as:

```text
tests/
├── conftest.py              <- app, client, auth_client, db fixtures
├── unit/
│   ├── test_models.py       <- User, Post, Comment model logic
│   └── test_utils.py        <- helper functions
├── integration/
│   ├── test_auth.py         <- login, logout, register flows
│   ├── test_posts.py        <- CRUD for blog posts
│   ├── test_comments.py     <- comment creation and moderation
│   └── test_errors.py       <- 404 and 500 handler pages
└── api/
    ├── test_posts_api.py    <- REST endpoints for posts
    └── test_auth_api.py     <- JWT token endpoints

Run with: pytest --cov=app --cov-report=html -v
Coverage: 91% across 220 statements
```

**What's tested:**
- All authentication flows (register, login, logout, password reset)
- All post CRUD operations including permission checks
- API endpoints with and without auth tokens
- 404/500 error pages render correctly
- Password hashing never stores plaintext

> ❓ **Is a JWT the same as a password?** No — a JWT is a *signed token* the server issues after login. On each request the client sends it back, and the server verifies the signature without touching the database. Treat it like a password though: anyone holding it can impersonate the user.

## 📋 Chapter Summary & Cheat Sheet

### conftest.py (fixtures)

Treat this section as a feedback loop: define an expectation, exercise the behavior, compare the result, and use the difference to learn. That framing keeps testing ideas concrete instead of procedural.

```python
@pytest.fixture
def app():
    app = create_app({"TESTING": True, "SQLALCHEMY_DATABASE_URI": "sqlite:///:memory:", "WTF_CSRF_ENABLED": False})
    with app.app_context():
        db.create_all()
        yield app
        db.drop_all()

@pytest.fixture
def client(app): return app.test_client()
```
### Test Patterns

Testing concepts stick better when you remember the purpose behind each step: create evidence that the behavior you care about still works, and learn quickly when it stops working.

```python
# GET route
def test_homepage(client):
    r = client.get("/")
    assert r.status_code == 200
    assert b"Welcome" in r.data

# POST form
def test_create(auth_client):
    r = auth_client.post("/posts/create",
        data={"title": "Test", "body": "Body"},
        follow_redirects=True)
    assert r.status_code == 200

# JSON API
def test_api(client):
    r = client.post("/api/posts", json={"title": "x"})
    assert r.status_code == 201
    assert r.json["title"] == "x"
```

### Coverage

```bash
pytest --cov=app --cov-report=html      # HTML report in htmlcov/
pytest --cov=app --cov-report=term-missing  # show missed lines
```
### Checklist

| Rule | Why |
|------|-----|
| Fresh DB per test (scope="function") | Prevents flaky tests |
| `WTF_CSRF_ENABLED = False` in test config | CSRF blocks POST submissions |
| `SQLALCHEMY_DATABASE_URI = "sqlite:///:memory:"` | Fast, isolated tests |
| Use `follow_redirects=True` for form POSTs | Avoid chasing 302s manually |
| Check DB state, not just response | Verify persistence |

> ❓ **What is CSRF?** A Cross-Site Request Forgery attack tricks a logged-in user's browser into sending a request to your site without their knowledge. A CSRF token — a secret value embedded in each form — ensures the submission really came from *your* page.

## 💪 Practice Prompts

**1. Test Your Routes**
Write a `conftest.py` with app/client fixtures using in-memory SQLite and CSRF disabled. Write tests for every route in your app: each GET returns 200, protected routes return 302 to login, and POST routes create the expected database records. Aim for 80%+ coverage.

**2. Authentication Test Suite**
Write a complete authentication test file testing: successful registration, duplicate email rejection, successful login, wrong password rejection, logout, and that protected routes redirect unauthenticated users. Use a helper function to create a test user to avoid repetition.

**3. Fix a Flaky Test**
Intentionally create a test that relies on another test's state (use `scope="module"` in your fixture). Run the tests multiple times with `pytest --randomly` to see it fail inconsistently. Then fix it by changing to `scope="function"` and using proper per-test setup. Document why the original was flaky.

<div style='width:100%; display:flex; justify-content:space-between; align-items:center; margin: 1em 0;'>
  <a href='15.%20error_handling_and_logging.ipynb' style='padding:6px 14px; background:#007bff; color:white; border-radius:4px; text-decoration:none; font-weight:bold;'>&#8592; Chapter 15: Error Handling</a>
  <a href='../TOC.md' style='padding:6px 14px; background:#6c757d; color:white; border-radius:4px; text-decoration:none; font-weight:bold;'>&#128218; Table of Contents</a>
  <a href='../7.%20deployment_and_production/17.%20caching_and_performance.ipynb' style='padding:6px 14px; background:#28a745; color:white; border-radius:4px; text-decoration:none; font-weight:bold;'>Chapter 17: Caching &#8594;</a>
</div>

<div style='width:100%; display:flex; justify-content:space-between; align-items:center; margin: 1em 0;'>
  <a href='15. error_handling_and_logging.ipynb' style='font-weight:bold; font-size:1.05em;'>&larr; Previous</a>
  <a href='../TOC.md' style='font-weight:bold; font-size:1.05em; text-align:center;'>Table of Contents</a>
  <a href='../7. deployment_and_production/17. caching_and_performance.ipynb' style='font-weight:bold; font-size:1.05em;'>Next &rarr;</a>
</div>